# Análise de E-commerce Brasileiro — Olist

Projeto de análise de dados com o dataset público da Olist (Kaggle), que reúne cerca de 100 mil pedidos feitos entre 2016 e 2018.

**Perguntas de negócio:**
- Como as vendas evoluíram ao longo do tempo?
- Quais estados e categorias concentram as vendas?
- Como os clientes pagam (tipo de pagamento e parcelas)?
- Atrasos na entrega afetam a avaliação do cliente?

**Ferramentas:** Python (pandas), SQL (SQLite) e Power BI.

## Leitura e exploração dos dados
Carregando as tabelas principais e verificando tamanho, tipos e valores nulos.

In [1]:
import pandas as pd
import sqlite3
import os

os.makedirs("dados_tratados", exist_ok=True)
conn = sqlite3.connect('olist.db')
dados = "dados/"

pedidos     = pd.read_csv(dados + "olist_orders_dataset.csv")
itens       = pd.read_csv(dados + "olist_order_items_dataset.csv")
pagamentos  = pd.read_csv(dados + "olist_order_payments_dataset.csv")
clientes    = pd.read_csv(dados + "olist_customers_dataset.csv")
produtos    = pd.read_csv(dados + "olist_products_dataset.csv")
avaliacoes  = pd.read_csv(dados + "olist_order_reviews_dataset.csv")
categorias  = pd.read_csv(dados + "product_category_name_translation.csv")


In [2]:
tabelas = {
    "pedidos": pedidos,
    "itens": itens,
    "pagamentos": pagamentos,
    "clientes": clientes,
    "produtos": produtos,
    "avaliacoes": avaliacoes,
    "categorias": categorias,
}

for nome, df in tabelas.items():
    print(f"{nome}: {df.shape[0]} linhas, {df.shape[1]} colunas")

pedidos: 99441 linhas, 8 colunas
itens: 112650 linhas, 7 colunas
pagamentos: 103886 linhas, 5 colunas
clientes: 99441 linhas, 5 colunas
produtos: 32951 linhas, 9 colunas
avaliacoes: 99224 linhas, 7 colunas
categorias: 71 linhas, 2 colunas


As tabelas de itens e de pagamentos têm mais linhas que a de pedidos, porque um mesmo pedido pode ter vários produtos e ser pago com mais de uma forma de pagamento.

### Explorando a tabela de pedidos
Verificando as primeiras linhas, os tipos das colunas, valores nulos, status dos pedidos e duplicatas.

In [3]:
pedidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [4]:
pedidos.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [5]:
pedidos["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

### Investigando os valores nulos nas datas de entrega
Verificando o status dos pedidos que não têm data de entrega ao cliente.

In [6]:
sem_entrega = pedidos[pedidos["order_delivered_customer_date"].isnull()]
sem_entrega["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [7]:
sem_entrega[sem_entrega["order_status"] == "delivered"]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


Os valores nulos nas datas de entrega correspondem a pedidos que não chegaram ao cliente (cancelados, em trânsito, indisponíveis), portanto são esperados. Alguns poucos pedidos com status `delivered` não têm data de entrega. São inconsistências e serão removidos na limpeza.

## Limpeza dos dados

### Conversão das datas
As colunas de data foram lidas como texto. Convertendo para o formato datetime para permitir cálculos com datas.

In [8]:
colunas_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for coluna in colunas_data:
    pedidos[coluna] = pd.to_datetime(pedidos[coluna])

pedidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


### Filtro de pedidos entregues
Mantendo apenas os pedidos entregues, pois prazo, faturamento e avaliação só fazem sentido para vendas concluídas. Os pedidos entregues sem data de entrega também são removidos.

In [9]:
entregues = pedidos[pedidos["order_status"] == "delivered"].copy()
entregues = entregues.dropna(subset=["order_delivered_customer_date"])

print(entregues.shape)

(96470, 8)


### Criação de novas colunas
- `dias_entrega`: dias entre a compra e a entrega ao cliente
- `atrasado`: indica se o pedido chegou depois da data prevista
- `mes_compra`: ano e mês da compra, para analisar as vendas ao longo do tempo

In [10]:
entregues["dias_entrega"] = (
    entregues["order_delivered_customer_date"] - entregues["order_purchase_timestamp"]
).dt.days

entregues["atrasado"] = (
    entregues["order_delivered_customer_date"].dt.normalize()
    > entregues["order_estimated_delivery_date"]
)

entregues["mes_compra"] = entregues["order_purchase_timestamp"].dt.to_period("M").astype(str)

In [11]:
entregues[["order_purchase_timestamp", "order_delivered_customer_date",
           "order_estimated_delivery_date", "dias_entrega", "atrasado", "mes_compra"]].head()

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,dias_entrega,atrasado,mes_compra
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8,False,2017-10
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13,False,2018-07
2,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9,False,2018-08
3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13,False,2017-11
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2,False,2018-02


In [12]:
print("Média de dias para entrega:", round(entregues["dias_entrega"].mean(), 1))
print("Percentual de pedidos atrasados:", round(entregues["atrasado"].mean() * 100, 1), "%")

Média de dias para entrega: 12.1
Percentual de pedidos atrasados: 6.8 %


**Resultado:** o prazo médio de entrega é de **12,1 dias**, e **6,8%** dos pedidos chegaram depois da data prevista.

### Itens e categorias de produtos
Cada linha da tabela de itens é um produto dentro de um pedido. Juntando com a tabela de produtos para obter a categoria de cada item.

In [13]:
itens.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [14]:
produtos["product_category_name"].isnull().sum()

610

In [15]:
produtos["product_category_name"] = produtos["product_category_name"].fillna("sem_categoria")

In [16]:
itens_categoria = itens.merge(
    produtos[["product_id", "product_category_name"]],
    on="product_id",
    how="left",
)

itens_categoria.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim


In [17]:
itens_entregues = itens_categoria[itens_categoria["order_id"].isin(entregues["order_id"])]

print(itens_entregues.shape)

(110189, 8)


### Pagamentos
Analisando as formas de pagamento e o número de parcelas.

In [18]:
pagamentos["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [19]:
pagamentos["payment_installments"].describe()

count    103886.000000
mean          2.853349
std           2.687051
min           0.000000
25%           1.000000
50%           1.000000
75%           4.000000
max          24.000000
Name: payment_installments, dtype: float64

In [20]:
pagamentos_entregues = pagamentos[
    (pagamentos["order_id"].isin(entregues["order_id"]))
    & (pagamentos["payment_type"] != "not_defined")
].copy()

nomes_pagamento = {
    "credit_card": "Cartão de crédito",
    "boleto": "Boleto",
    "voucher": "Voucher",
    "debit_card": "Cartão de débito",
}

pagamentos_entregues["tipo_pagamento"] = pagamentos_entregues["payment_type"].map(nomes_pagamento)

pagamentos_entregues["tipo_pagamento"].value_counts()

tipo_pagamento
Cartão de crédito    74579
Boleto               19191
Voucher               5493
Cartão de débito      1485
Name: count, dtype: int64

### Avaliações
Alguns pedidos receberam mais de uma avaliação. Calculando a nota média por pedido para ter uma única nota por pedido.

In [21]:
avaliacoes["order_id"].duplicated().sum()

551

In [22]:
nota_pedido = avaliacoes.groupby("order_id")["review_score"].mean().reset_index()

nota_pedido.head()

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


### Montando a tabela final de pedidos
Adicionando à tabela de pedidos entregues o estado do cliente e a nota da avaliação.

In [23]:
print("Antes:", entregues.shape)

entregues = entregues.merge(
    clientes[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
    on="customer_id",
    how="left",
)

entregues = entregues.merge(nota_pedido, on="order_id", how="left")

print("Depois:", entregues.shape)

Antes: (96470, 11)
Depois: (96470, 15)


In [24]:
entregues.isnull().sum()

order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 14
order_delivered_carrier_date       1
order_delivered_customer_date      0
order_estimated_delivery_date      0
dias_entrega                       0
atrasado                           0
mes_compra                         0
customer_unique_id                 0
customer_city                      0
customer_state                     0
review_score                     646
dtype: int64

In [25]:
pagamentos_entregues.loc[pagamentos_entregues["payment_installments"] == 0, "payment_installments"] = 1

pagamentos_entregues["payment_installments"].describe()

count    100748.000000
mean          2.851660
std           2.684435
min           1.000000
25%           1.000000
50%           1.000000
75%           4.000000
max          24.000000
Name: payment_installments, dtype: float64

**Decisões sobre a qualidade dos dados:**
- Pagamentos com tipo `not_defined` (3 registros) foram removidos.
- Pagamentos com 0 parcelas foram considerados como 1 parcela (pagamento à vista), já que todo pagamento tem ao menos uma parcela.
- 14 pedidos entregues sem data de aprovação e 1 sem data de envio à transportadora foram mantidos, pois essas colunas não são usadas na análise.
- 646 pedidos não possuem avaliação e foram desconsiderados apenas no cálculo da nota média.

In [26]:
entregues.to_sql("entregues", conn, if_exists="replace", index=False)
itens_entregues.to_sql("itens", conn, if_exists="replace", index=False)
pagamentos_entregues.to_sql("pagamentos", conn, if_exists="replace", index=False)

100748

### Visão geral dos pedidos entregues

In [27]:
consulta = """
SELECT
    COUNT(*) AS total_pedidos,
    ROUND(AVG(dias_entrega), 1) AS media_dias_entrega,
    ROUND(AVG(atrasado) * 100, 1) AS pct_atrasados,
    ROUND(AVG(review_score), 2) AS nota_media
FROM entregues
"""

pd.read_sql_query(consulta, conn)

,total_pedidos,media_dias_entrega,pct_atrasados,nota_media
0,96470,12.1,6.8,4.16


### Evolução das vendas por mês
Quantidade de pedidos e faturamento (soma dos pagamentos) por mês de compra.

In [28]:
consulta = """
SELECT
    e.mes_compra,
    COUNT(DISTINCT e.order_id) AS pedidos,
    ROUND(SUM(p.payment_value), 2) AS faturamento
FROM entregues AS e
JOIN pagamentos AS p
    ON e.order_id = p.order_id
GROUP BY e.mes_compra
ORDER BY e.mes_compra
"""

vendas_mes = pd.read_sql_query(consulta, conn)
vendas_mes

,mes_compra,pedidos,faturamento
0,2016-10,265,46566.71
1,2016-12,1,19.62
2,2017-01,750,127545.67
3,2017-02,1653,271298.65
4,2017-03,2546,414369.39
5,2017-04,2303,390952.18
6,2017-05,3545,566872.73
7,2017-06,3135,490225.60
8,2017-07,3872,566403.93
9,2017-08,4193,646000.61


**Observações:**
- Os resultados de prazo médio e percentual de atraso batem com os calculados em pandas, validando a análise.
- 2016 tem dados incompletos (apenas outubro e dezembro, com poucos pedidos). Por isso, as análises temporais consideram o período de janeiro/2017 a agosto/2018.
- O pico de pedidos ocorre em novembro/2017 (7.288 pedidos, 63% acima de outubro), provavelmente por causa da Black Friday.
- As vendas crescem ao longo de 2017 e se estabilizam entre 6 e 7 mil pedidos por mês em 2018.

### Estados com mais pedidos

In [29]:
consulta = """
SELECT
    customer_state AS estado,
    COUNT(*) AS pedidos,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM entregues), 1) AS pct_pedidos
FROM entregues
GROUP BY customer_state
ORDER BY pedidos DESC
LIMIT 10
"""

pd.read_sql_query(consulta, conn)

,estado,pedidos,pct_pedidos
0,SP,40494,42.0
1,RJ,12350,12.8
2,MG,11354,11.8
3,RS,5344,5.5
4,PR,4923,5.1
5,SC,3546,3.7
6,BA,3256,3.4
7,DF,2080,2.2
8,ES,1995,2.1
9,GO,1957,2.0


VENDAS MUITO CONCENTRADAS NO SUDESTE (SP+RJ+MG = 66,6% DAS VENDAS)

### Categorias com maior faturamento
Faturamento calculado pela soma dos preços dos produtos (sem frete).

In [30]:
consulta = """
SELECT
    product_category_name AS categoria,
    COUNT(*) AS itens_vendidos,
    ROUND(SUM(price), 2) AS faturamento
FROM itens
GROUP BY product_category_name
ORDER BY faturamento DESC
LIMIT 10
"""

pd.read_sql_query(consulta, conn)

,categoria,itens_vendidos,faturamento
0,beleza_saude,9465,1233131.72
1,relogios_presentes,5857,1165898.98
2,cama_mesa_banho,10953,1023434.76
3,esporte_lazer,8430,954673.55
4,informatica_acessorios,7643,888613.62
5,moveis_decoracao,8160,711927.69
6,utilidades_domesticas,6795,615628.69
7,cool_stuff,3718,610204.10
8,automotivo,4139,578849.35
9,brinquedos,4029,471097.49


### Formas de pagamento
Quantidade de pagamentos, participação e ticket médio por forma de pagamento.

In [31]:
consulta = """
SELECT
    tipo_pagamento,
    COUNT(*) AS qtd_pagamentos,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM pagamentos), 1) AS pct_pagamentos,
    ROUND(SUM(payment_value), 2)    AS valor_total,
    ROUND(AVG(payment_value), 2)    AS ticket_medio
FROM pagamentos
GROUP BY tipo_pagamento
ORDER BY qtd_pagamentos DESC
"""

pd.read_sql_query(consulta, conn)

,tipo_pagamento,qtd_pagamentos,pct_pagamentos,valor_total,ticket_medio
0,Cartão de crédito,74579,74.0,12100070.20,162.25
1,Boleto,19191,19.0,2769932.58,144.33
2,Voucher,5493,5.5,343013.19,62.45
3,Cartão de débito,1485,1.5,208066.88,140.11


### Parcelamento no cartão de crédito

In [32]:
consulta = """
SELECT
    payment_installments AS parcelas,
    COUNT(*) AS qtd_pagamentos,
    ROUND(AVG(payment_value), 2) AS valor_medio
FROM pagamentos
WHERE tipo_pagamento = 'Cartão de crédito'
GROUP BY payment_installments
ORDER BY parcelas
"""

parcelas = pd.read_sql_query(consulta, conn)
parcelas

,parcelas,qtd_pagamentos,valor_medio
0,1,24759,95.60
1,2,12075,126.59
2,3,10162,142.00
3,4,6889,163.63
4,5,5094,182.32
5,6,3804,208.54
6,7,1563,185.95
7,8,4136,306.29
8,9,618,197.97
9,10,5150,410.54


**Observações:**
- **Estados:** as vendas são concentradas no Sudeste. SP representa 42% dos pedidos, e SP, RJ e MG juntos somam 66,6%.
- **Categorias:** beleza_saude lidera em faturamento, enquanto cama_mesa_banho lidera em quantidade de itens vendidos. relogios_presentes vende menos itens, mas fatura mais, indicando produtos de maior valor.
- **Formas de pagamento:** o cartão de crédito responde por 74% dos pagamentos, seguido pelo boleto (19%), um meio de pagamento local relevante no Brasil. O voucher tem o menor ticket médio (R$ 62,45), por geralmente cobrir apenas parte da compra.
- **Parcelamento:** o pagamento em 1x é o mais comum, com destaque para 10x, típico do "10x sem juros". Em geral, quanto maior o valor da compra, maior o número de parcelas.

### Atraso na entrega x avaliação do cliente

In [33]:
consulta = """
SELECT
    CASE
        WHEN atrasado = 1 THEN 'Atrasado'
        ELSE 'No prazo'
    END AS situacao_entrega,
    COUNT(*) AS pedidos,
    ROUND(AVG(review_score), 2) AS nota_media
FROM entregues
GROUP BY situacao_entrega
"""

pd.read_sql_query(consulta, conn)

,situacao_entrega,pedidos,nota_media
0,Atrasado,6534,2.27
1,No prazo,89936,4.29


## 4. Exportação dos dados para o Power BI
Exportando as tabelas tratadas, considerando o período de janeiro/2017 a agosto/2018 (2016 tem dados incompletos).

In [34]:

pedidos_pbi = entregues[entregues["mes_compra"] >= "2017-01"].copy()

pedidos_pbi["data_compra"] = pedidos_pbi["order_purchase_timestamp"].dt.date
pedidos_pbi["situacao_entrega"] = pedidos_pbi["atrasado"].map({True: "Atrasado", False: "No prazo"})
pedidos_pbi["atrasado"] = pedidos_pbi["atrasado"].astype(int)

pedidos_pbi = pedidos_pbi[[
    "order_id", "customer_unique_id", "customer_state", "customer_city",
    "data_compra", "mes_compra", "dias_entrega", "atrasado",
    "situacao_entrega", "review_score",
]].rename(columns={
    "order_id": "id_pedido",
    "customer_unique_id": "id_cliente",
    "customer_state": "estado",
    "customer_city": "cidade",
    "review_score": "nota_avaliacao",
})

print(pedidos_pbi.shape)
pedidos_pbi.head()

(96203, 10)


,id_pedido,id_cliente,estado,cidade,data_compra,mes_compra,dias_entrega,atrasado,situacao_entrega,nota_avaliacao
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,SP,sao paulo,2017-10-02,2017-10,8,0,No prazo,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,BA,barreiras,2018-07-24,2018-07,13,0,No prazo,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,GO,vianopolis,2018-08-08,2018-08,9,0,No prazo,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,RN,sao goncalo do amarante,2017-11-18,2017-11,13,0,No prazo,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,SP,santo andre,2018-02-13,2018-02,2,0,No prazo,5.0


In [35]:
itens_pbi = itens_entregues[itens_entregues["order_id"].isin(pedidos_pbi["id_pedido"])]

itens_pbi = itens_pbi[["order_id", "product_category_name", "price", "freight_value"]].rename(columns={
    "order_id": "id_pedido",
    "product_category_name": "categoria",
    "price": "preco",
    "freight_value": "frete",
})

pagamentos_pbi = pagamentos_entregues[pagamentos_entregues["order_id"].isin(pedidos_pbi["id_pedido"])]

pagamentos_pbi = pagamentos_pbi[["order_id", "tipo_pagamento", "payment_installments", "payment_value"]].rename(columns={
    "order_id": "id_pedido",
    "payment_installments": "parcelas",
    "payment_value": "valor_pago",
})

print(itens_pbi.shape, pagamentos_pbi.shape)

(109872, 4) (100465, 4)


In [36]:
pedidos_pbi.to_csv("dados_tratados/pedidos.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
itens_pbi.to_csv("dados_tratados/itens.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
pagamentos_pbi.to_csv("dados_tratados/pagamentos.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")